In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error
from scipy.spatial.distance import cdist
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def mmd_loss(y_true, y_pred, sigma=1.0):
    n_true = y_true.shape[0]
    n_pred = y_pred.shape[0]
    
    XX = cdist(y_true, y_true, 'sqeuclidean')
    YY = cdist(y_pred, y_pred, 'sqeuclidean')
    XY = cdist(y_true, y_pred, 'sqeuclidean')
    
    K_XX = np.exp(-XX / (2 * sigma**2))
    K_YY = np.exp(-YY / (2 * sigma**2))
    K_XY = np.exp(-XY / (2 * sigma**2))
    
    mmd = (K_XX.sum() - np.trace(K_XX)) / (n_true * (n_true - 1)) + \
          (K_YY.sum() - np.trace(K_YY)) / (n_pred * (n_pred - 1)) - \
          2 * K_XY.sum() / (n_true * n_pred)
    
    return max(mmd, 0)

class RidgeMMD(Ridge):
    def __init__(self, alpha=100, fit_intercept=True, a=1.0, b=0.1, sigma=1.0, **kwargs):
        super().__init__(alpha=alpha, fit_intercept=fit_intercept, **kwargs)
        self.a = a
        self.b = b
        self.sigma = sigma
    
    def fit(self, X, y, sample_weight=None):
        super().fit(X, y, sample_weight)
        return self
    
    def _loss(self, X, y):
        y_pred = self.predict(X)
        mse = np.mean((y - y_pred) ** 2)
        mmd = mmd_loss(y, y_pred, self.sigma)
        return self.a * mse + self.b * mmd

In [ ]:
diff_matrix = pd.read_csv("/jdata/qmy/VirtualCell/diff_matrix_predict/sc_pseudo_bulk_10_diff_5.0uM.csv", index_col=[0,1])
diff_matrix_200 = pd.read_csv("/jdata/qmy/VirtualCell/diff_matrix_predict/sc_pseudo_bulk_10_diff_5.0uM_top200.csv", index_col=[0,1])

In [ ]:
common_samples = diff_matrix.index.intersection(diff_matrix_200.index)
print(f"共同样本数: {len(common_samples)}")
drug_names = [idx[0] for idx in common_samples]
cell_lines = [idx[1] for idx in common_samples]

unique_cells = list(set(cell_lines))
print(f"细胞系数: {len(unique_cells)}")
cell_models = {}
cell_results = {}
all_predictions = []
all_indices = []

In [ ]:
# 存储所有预测结果
all_predictions = []
all_indices = []

for cell_line in unique_cells:
    cell_mask = np.array([cl == cell_line for cl in cell_lines])
    cell_samples = np.where(cell_mask)[0]
    
    if len(cell_samples) < 100:
        continue
    
    X_cell = diff_matrix_200.loc[common_samples].iloc[cell_samples].values
    Y_cell = diff_matrix.loc[common_samples].iloc[cell_samples].values
    drugs_cell = [drug_names[i] for i in cell_samples]
    unique_drugs_cell = list(set(drugs_cell))
    
    if len(unique_drugs_cell) < 80:
        continue
    
    np.random.seed(42)
    test_drugs = np.random.choice(unique_drugs_cell, min(75, len(unique_drugs_cell)//5), replace=False)
    train_drugs = [d for d in unique_drugs_cell if d not in test_drugs]
    
    train_mask = np.array([d in train_drugs for d in drugs_cell])
    test_mask = np.array([d in test_drugs for d in drugs_cell])
    
    X_train, X_test = X_cell[train_mask], X_cell[test_mask]
    Y_train, Y_test = Y_cell[train_mask], Y_cell[test_mask]
    ridge_mmd = RidgeMMD(alpha=100.0, fit_intercept=True, a=100.0, b=0.0, sigma=1.0, random_state=42, solver='auto')
    ridge_mmd.fit(X_train, Y_train)
    Y_pred = ridge_mmd.predict(X_test)
    r2 = r2_score(Y_test, Y_pred, multioutput="variance_weighted")
    mse = mean_squared_error(Y_test, Y_pred)
    
    cell_models[cell_line] = ridge_mmd
    cell_results[cell_line] = {
        'train_samples': X_train.shape[0],
        'test_samples': X_test.shape[0],
        'train_drugs': len(train_drugs),
        'test_drugs': len(test_drugs),
        'r2': r2,
        'mse': mse
    }
    test_indices = [common_samples[i] for i in np.array(cell_samples)[test_mask]]
    for i, idx in enumerate(test_indices):
        all_predictions.append(Y_pred[i])
        all_indices.append(idx)
    
    print(f"{cell_line}: R²={r2:.4f}, MSE={mse:.6f}")

# 创建预测矩阵
if all_predictions:
    pred_matrix = pd.DataFrame(
        np.array(all_predictions),
        index=pd.MultiIndex.from_tuples(all_indices, names=['drug', 'cell']),
        columns=diff_matrix.columns
    )
    
    # 保存
    pred_matrix.to_csv("/jdata/qmy/VirtualCell/diff_matrix_predict/sc_predictions_5.0uM.ipynb")
    print(f"\n预测矩阵已保存: {pred_matrix.shape}")
else:
    print("没有预测结果保存")

